# Stage 2 Notebook 45 - Exp2PP Anchor width 1.0 + 30 epochs scale-up

**The capacity hypothesis.** Every loss tweak (NB39 focal, NB40 ASL, NB41 lineiou+QFL, NB42 dual head) plateaued at oracle_f1 ~ 0.45 and decoded_f1 ~ 0.04. The model has been width=0.5, embed_dim=128, 20 epochs across all four runs. With matched_iou and oracle_f1 still climbing at epoch 20 in NB41 (0.490 -> 0.508 in the last 5 epochs) and NB40 (0.479 -> 0.483), the model is undertrained AND underweight.

Exp2PP scales up two axes simultaneously while keeping NB40's stable loss recipe:

- `model.width: 0.5 -> 1.0` (full RMT-PPAD width)
- `lane_head.embed_dim: 128 -> 192` (larger per-prior feature so cls has more discriminative capacity)
- `lane_head.roi_mid_channels: 48 -> 64`
- `train.end_epoch: 20 -> 30` (cosine LR amortizes a longer decay window)
- `train.lr_scheduler.warmup_epochs: 2 -> 3`

Loss / matching / dataset are unchanged from NB40 (Exp2KK). This isolates the capacity question.

GPU mem: NB40 used 10.4 GB / 95.6 GB peak. Width 1.0 + embed 192 should land ~ 25-35 GB, well within budget.

### Run mode

1. Keep `DEBUG_MODE = True` for the first run -- the wider model needs a smoke check.
2. After smoke + debug pass, change to `False` for the 30-epoch short run.
3. AMP keeps wall-clock ~ 45 minutes for 30 epochs at 3000 samples (estimated; ~ 50% slower than NB40).
4. Output mirrored to notebook cell, Colab runtime log, Drive log file.
5. Do not rerun NB00. Independent of any prior NB; only depends on the dataset tar.

In [1]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [2]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp40_rmt_gca_anchor_width1_long30_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp40_rmt_gca_anchor_width1_long30_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp40_rmt_gca_anchor_width1_long30_joint_smoke.log
OK exp40_rmt_gca_anchor_width1_long30_joint.yaml
  lane_shape=(1, 16, 72, 2) det_shape=(1, 4, 4)
  lane_loss=5.6411 det_loss=3.5817 grad_cos=0.1788 lambda_lane=0.0500
  gate_stats={'gate/det_mean': 0.5034533143043518, 'gate/lane_mean': 0.5029304623603821, 'gate/det_sat_low': 0.0, 'gate/det_sat_high': 0.0, 'gate/lane_sat_low': 0.0, 'gate/lane_sat_high': 0.0}
[run_streaming] return_code=0


0

In [3]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp40_rmt_gca_anchor_width1_long30_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'short30'
    EPOCHS = 30
    BATCH_SIZE = 8
    LIMIT_TRAIN = 3000
    LIMIT_VAL = 1000
    PRINT_EVERY = 5

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-train', str(LIMIT_TRAIN),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

流式输出内容被截断，只能显示最后 5000 行内容。
epoch=8 step=190/375 total=2.1496 det=1.9749 lane=3.9319 lambda_lane=1.0000 cls=0.0890 reg=0.1405 iou=0.5915 grad_cos=nan speed=4.60step/s eta_min=0.7
epoch=8 step=195/375 total=2.1607 det=2.0794 lane=3.8600 lambda_lane=1.0000 cls=0.0879 reg=0.1436 iou=0.5598 grad_cos=nan speed=4.60step/s eta_min=0.7
epoch=8 step=200/375 total=2.2975 det=2.4260 lane=4.0360 lambda_lane=1.0000 cls=0.0860 reg=0.1400 iou=0.6042 grad_cos=-0.0053 speed=4.59step/s eta_min=0.6
epoch=8 step=205/375 total=2.2844 det=2.3710 lane=4.0471 lambda_lane=1.0000 cls=0.0858 reg=0.1443 iou=0.6126 grad_cos=nan speed=4.59step/s eta_min=0.6
epoch=8 step=210/375 total=2.2155 det=2.2416 lane=3.9092 lambda_lane=1.0000 cls=0.0870 reg=0.1328 iou=0.5893 grad_cos=nan speed=4.59step/s eta_min=0.6
epoch=8 step=215/375 total=2.2049 det=1.9659 lane=4.1861 lambda_lane=1.0000 cls=0.0888 reg=0.1466 iou=0.6499 grad_cos=nan speed=4.60step/s eta_min=0.6
epoch=8 step=220/375 total=2.2081 det=1.9920 lane=4.1717 lambda

0

## What to watch in Exp2PP training

Pass criteria at epoch 30:
- **`val/matched_line_iou >= 0.55`** (NB41 hit 0.508 with width 0.5; doubling capacity + 50% more epochs should add 0.05).
- **`val/lane/decoded_oracle_f1 >= 0.50`** (NB41 oracle was 0.459 still climbing).
- **`val/lane/decoded_f1 (cls_x_mask) >= 0.15`**.
- `train_total` curve should still be decreasing at epoch 30 (no plateau) -- if it plateaus by epoch 20, capacity was the bottleneck and we should retrain with width 1.5.

Failure signals:
- oracle_f1 plateau at 0.46 by epoch 30: capacity was NOT the bottleneck; the architecture itself can't improve geometry past this point on width-0.5-equivalent representations.
- pos-neg gap still < 0.02: even at width 1.0, the cls task is fundamentally unsolvable by per-prior ROI features. (Confirms Exp2NN's mask-consistency direction.)
- GPU OOM: drop batch_size to 4 or set width back to 0.75.